# 00 — Setup & Environment Checks
## BTC Data Collection Pipeline v2

Ce notebook vérifie l'environnement RunPod, les connexions GCS et Bitcoin Core RPC,
l'espace disque disponible, et affiche l'état du pipeline.

**v2** : ajout de la gestion de Bitcoin Core (démarrage, monitoring sync).

**À exécuter en premier, avant tout autre notebook.**

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 1 — Installation des dépendances
# ════════════════════════════════════════════════════════════════════════
import subprocess, sys

PACKAGES = [
    "polars", "pandas", "pyarrow", "google-cloud-storage", "gcsfs",
    "aiohttp", "websockets", "python-bitcoinrpc", "loguru", "tqdm",
    "ipywidgets", "python-dateutil", "requests",
]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ Tous les packages installés")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 2 — Variables d'environnement
# ════════════════════════════════════════════════════════════════════════
import os

# ⚠️  MODIFIER CES VALEURS SELON VOTRE CONFIGURATION
os.environ.setdefault("GCS_BUCKET_NAME",              "btc-training-data")
os.environ.setdefault("GOOGLE_CLOUD_PROJECT",          "mon-projet-gcp")
os.environ.setdefault("GOOGLE_APPLICATION_CREDENTIALS", "/workspace/service-account.json")
os.environ.setdefault("STORAGE_BACKEND",               "gcs")       # "gcs" ou "local"
os.environ.setdefault("WORKSPACE_DIR",                 "/workspace")
os.environ.setdefault("BTC_RPC_USER",                  "btcuser")
os.environ.setdefault("BTC_RPC_PASSWORD",              "secret")
os.environ.setdefault("BTC_RPC_HOST",                  "127.0.0.1")
os.environ.setdefault("BTC_RPC_PORT",                  "8332")
os.environ.setdefault("GLASSNODE_API_KEY",             "")           # optionnel

print("✅ Variables d'environnement définies")
for k in ["GCS_BUCKET_NAME", "STORAGE_BACKEND", "WORKSPACE_DIR"]:
    print(f"   {k} = {os.environ.get(k)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 3 — Vérification de l'espace disque
# ════════════════════════════════════════════════════════════════════════
import shutil

workspace = os.environ.get("WORKSPACE_DIR", "/workspace")
if os.path.exists(workspace):
    total, used, free = shutil.disk_usage(workspace)
    print(f"📁 Disque ({workspace}):")
    print(f"   Total  : {total/1e9:.1f} GB")
    print(f"   Utilisé: {used/1e9:.1f} GB")
    print(f"   Libre  : {free/1e9:.1f} GB")
    if free < 80e9:
        print("🚨 ATTENTION : moins de 80 GB libres — espace insuffisant !")
    elif free < 150e9:
        print("⚠️  ATTENTION : moins de 150 GB libres — réduire les downloads parallèles")
    else:
        print("✅ Espace disque OK")
else:
    print(f"⚠️  Le dossier {workspace} n'existe pas — vérifier le Network Volume RunPod")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 4 — Vérification de la connexion GCS
# ════════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "/workspace")

from btc_pipeline.storage.gcs_client import StorageClient

try:
    storage = StorageClient()

    # Test upload/download
    import pandas as pd
    test_df = pd.DataFrame({"test": [1, 2, 3]})
    storage.stream_upload_parquet(test_df, "metadata/_test_connection.parquet", skip_if_exists=False)
    result = storage.download_parquet("metadata/_test_connection.parquet")
    assert len(result) == 3, "Round-trip test failed"
    print(f"✅ Connexion GCS OK (backend={storage.backend}, bucket={storage.bucket_name})")
except Exception as e:
    print(f"❌ Erreur GCS : {e}")
    print("   Vérifier : service-account.json, GOOGLE_APPLICATION_CREDENTIALS, bucket name")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 5 — Vérification de la connexion Bitcoin Core RPC
# ════════════════════════════════════════════════════════════════════════
try:
    from btc_pipeline.collectors.bitcoin_core import get_rpc, get_chain_info
    rpc = get_rpc()
    info = get_chain_info(rpc)
    print(f"✅ Bitcoin Core RPC OK")
    print(f"   Chain          : {info['chain']}")
    print(f"   Blocks         : {info['blocks']:,}")
    print(f"   Headers        : {info['headers']:,}")
    print(f"   Verification   : {info.get('verificationprogress', 0)*100:.2f}%")
    print(f"   Size on disk   : {info.get('size_on_disk', 0)/1e9:.1f} GB")
    synced = info['blocks'] == info['headers']
    print(f"   {'✅ Noeud synchronise' if synced else '⏳ Synchronisation en cours...'}")
except Exception as e:
    print(f"⚠️  Bitcoin Core RPC non disponible : {e}")
    print("   Lancer : bitcoind -datadir=/workspace/.bitcoin -daemon")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 6 — État du pipeline (pipeline_state.json)
# ════════════════════════════════════════════════════════════════════════
import json

state = storage.get_pipeline_state()
print("📋 État du pipeline :")
print(json.dumps(state, indent=2, default=str))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 7 — Gestion Bitcoin Core (v2)
# ════════════════════════════════════════════════════════════════════════
import subprocess
import time

BITCOIN_DATADIR = "/workspace/.bitcoin"
RPC_PASSWORD = os.environ.get("BTC_RPC_PASSWORD", "btcpassword_change_me")

def start_bitcoind():
    """Demarre le daemon Bitcoin Core si pas deja actif."""
    try:
        result = subprocess.run(
            ["bitcoin-cli", f"-datadir={BITCOIN_DATADIR}", "getblockchaininfo"],
            capture_output=True, timeout=10
        )
        if result.returncode == 0:
            print("✅ Bitcoin Core deja actif")
            return True
    except Exception:
        pass

    print("▶ Demarrage de Bitcoin Core...")
    subprocess.Popen([
        "bitcoind",
        f"-datadir={BITCOIN_DATADIR}",
        "-daemon"
    ])
    time.sleep(5)

    # Verifier que le demarrage a reussi
    for attempt in range(12):  # Attendre jusqu'a 60s
        try:
            result = subprocess.run(
                ["bitcoin-cli", f"-datadir={BITCOIN_DATADIR}", "getblockchaininfo"],
                capture_output=True, timeout=10
            )
            if result.returncode == 0:
                print("✅ Bitcoin Core demarre")
                return True
        except Exception:
            pass
        time.sleep(5)
        print(f"  En attente ({attempt+1}/12)...")

    print("❌ Impossible de demarrer Bitcoin Core")
    return False

def get_sync_progress():
    """Retourne la progression de la synchronisation."""
    try:
        result = subprocess.run(
            ["bitcoin-cli", f"-datadir={BITCOIN_DATADIR}", "getblockchaininfo"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            import json
            info = json.loads(result.stdout)
            return {
                "blocks": info["blocks"],
                "headers": info["headers"],
                "progress": info.get("verificationprogress", 0) * 100,
                "synced": info["blocks"] == info["headers"],
                "size_gb": info.get("size_on_disk", 0) / 1e9,
            }
    except Exception as e:
        return {"error": str(e)}

# Demarrer et afficher le status
started = start_bitcoind()
if started:
    sync = get_sync_progress()
    if "error" not in sync:
        print(f"  Blocs      : {sync['blocks']:,} / {sync['headers']:,}")
        print(f"  Progression: {sync['progress']:.2f}%")
        print(f"  Taille     : {sync['size_gb']:.1f} GB")
        if not sync['synced']:
            remaining_blocks = sync['headers'] - sync['blocks']
            est_hours = remaining_blocks * 0.15 / 3600
            print(f"  ⏳ Synchronisation en cours — ~{est_hours:.0f}h restantes")
            print(f"  Le telechargement Binance peut etre lance en parallele")

# Test internet (Binance)
import requests
try:
    r = requests.head("https://data.binance.vision", timeout=10)
    print(f"✅ Connexion a data.binance.vision OK (status={r.status_code})")
except Exception as e:
    print(f"❌ Pas de connexion a Binance : {e}")

# Test mempool.space
try:
    r = requests.get("https://mempool.space/api/blocks/tip/height", timeout=10)
    print(f"✅ Connexion a mempool.space OK (block height: {r.text})")
except Exception as e:
    print(f"⚠️  mempool.space non accessible : {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Cellule 8 — Resume
# ════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("RESUME — PRET POUR LE LANCEMENT DU PIPELINE v2")
print("=" * 60)
print(f"  Backend stockage : {storage.backend}")
print(f"  Espace libre     : {free/1e9:.1f} GB" if 'free' in dir() else "  Espace libre     : N/A")
print(f"  Notebooks a lancer :")
print(f"    1. 01_binance_historical.ipynb  (24-36h)")
print(f"    2. 02_blockchain_blocks.ipynb   (24-48h, parallele)")
print(f"    3. 04_websocket_live.ipynb      (continu, IMMEDIATEMENT)")
print(f"    4. 03_mempool_and_onchain.ipynb  (~1-2h)")
print(f"    5. 05_dataset_builder.ipynb     (apres toutes les sources)")
print("=" * 60)